# Payment Summary Cell Audit — solution

Reconstruct, for each receipt photo: the number of purchased item rows, the printed final
total **as printed**, the four summary-field presence flags, and a 4×4 semantic cell map.

```
row_score = 0.45·semantic_grid + 0.35·total_price + 0.12·num_items + 0.08·fields
```

Everything below is trained from scratch on the 529 public training images plus
**procedurally rendered text images** built from the font files that ship inside the locally
installed `matplotlib` package. Every weight is initialised randomly and fitted during the
run: the solution reads no model files, uses no outside corpus, makes no network request,
executes no remote code, and contains no per-id answer table.

**Runnable artefact:** `solution.py` — `python3 solution.py <public_dir> <submission_out.csv>`.
This notebook is documentation and a driver for that script.

## 1. What the labels actually encode

Inspecting the training answers pins down the structure of the task:

| observation | consequence |
|---|---|
| grid row 0 is `0` in ~99% of images, row 3 in ~70% | the receipt's summary block sits in the middle vertical band |
| value `3` occupies *more* cells than value `1` | overlapping text blocks resolve to the **largest** class, so the final-total block masks item cells it shares a cell with |
| `num_items` ∈ {2,3,4} for 80% of rows | the count is heavily peaked but still needs real evidence to beat the mode |
| `total_price` is `DD,DDD` or `DD.DDD` for ~70% of rows, 345 distinct values in 529 rows | the separator character must be reproduced exactly — a digit-only reader is not enough |

A best-possible **constant** submission scores **0.333** (grid 0.446 / total 0.179 /
num_items 0.312 / fields 0.405), so every component needs a real model.

## 2. Method

**Stage A — layout network (grid, count, fields).**
A small convolutional network reads the whole photo at 448×320 with five input planes
(RGB, an illumination-normalised grey plane, and a binarised text-ink plane). The grid head is
*spatially aligned*: the feature map is average-pooled to exactly 4×4 and a 1×1 convolution emits
the four class logits per cell, so cell *(r,c)* is predicted from the image region it covers.
Two global heads predict the item count and the four field flags. Trained 5-fold, three times
(70/130/130 epochs, two widths), photometric + small-affine augmentation only — layout must
not be flipped.

**Stage B — candidate strips.**
The predicted grid gives a region of interest. Inside it the page is deskewed (the rotation
that maximises the sharpness of the horizontal ink projection, searched over ±8°), binarised
adaptively at a block size tied to the estimated glyph height, and grouped into text lines.
Each line yields **several competing crops** — the whole line, three different word-gap
thresholds, and the right-hand remainder — because a single segmentation rule silently loses
the answer on blurred or tightly-set receipts. Multi-hypothesis extraction lifted the fraction
of images in which the true total is *readable at all* from 0.32 to 0.48.

**Stage C — sequence reader.**
A convolutional-recurrent network with a connectionist-temporal-classification loss reads every
strip over a 76-symbol alphabet (digits, letters, punctuation). It is fitted from random
initialisation on rendered strings — receipt vocabulary, random words and money patterns,
drawn in 176 distinct locally installed fonts — pushed through a degradation chain whose most
important element is a **resolution bottleneck**: render large, downsample to a random 7–30 px
line height, upsample back to 32 px. Without it the model learns to read sharp text and fails
on real crops. Fonts are screened for glyph distinctness first: one font rendered every digit
as an identical box and silently poisoned the CTC targets.

**Stage D — answer selection, in two stages.**
Every decoded string and every numeric substring becomes a candidate, plus its **canonical
regroupings** — these receipts group thousands, so once the digits are right the printed form
is recoverable deterministically (`25000` → `25.000`, `25,000`). Selection then splits in two,
because *which digits the receipt shows* and *how they were printed* are different questions:
stage A ranks digit strings (all candidate rows sharing a digit string are pooled into one),
stage B picks the printed form among the survivors using the receipt's own formatting habits.
Stage B is trained on stage A's **out-of-fold** picks.

Stage A additionally sees **receipt-arithmetic features**: a receipt is a self-consistent
ledger, so the total is the value that equals subtotal + tax + service, and the one that cash
minus change lands on. These relations hold between candidate *values* and so need no labels.
Worth +0.007 on `total_price`.

**Stage E — grid refinement, count and flags.**
The layout network cannot read, but "subtotal area" (class 2) versus "final-total area"
(class 3) is a semantic distinction carried by words like SUBTOTAL / PPN / SERVICE versus
TOTAL. A per-cell gradient-boosted classifier takes the network's own probabilities, those of
the four neighbouring cells, and fuzzy keyword evidence for text falling in that cell; this
moves the grid from 0.741 to 0.766. Counts and flags blend image-level statistics with the
network's probabilities. Two structural constraints are applied at decode: class 3 occurs in
100% of training grids, and class 2 never occurs when no field flag is set.

## 3. Cross-validated results (529 training rows, 5-fold)

| component | weight | out-of-fold |
|---|---|---|
| `semantic_grid` | 0.45 | **0.7658** |
| `total_price`   | 0.35 | **0.6029** (exact-string 0.5312) |
| `num_items`     | 0.12 | **0.4896** |
| `fields_present`| 0.08 | **0.7482** |
| **row score**   |      | **0.6742** |

For reference: best constant submission 0.333, stated baseline 0.58.

### Where the remaining loss is

`total_price` decomposes cleanly. The correct printed string is among the generated
candidates for **0.6465** of images; of those, the two-stage selector returns it **82%** of
the time (stage A picks the right *digits* at 0.906 efficiency, stage B then recovers the
printed form 90.6% of the time). So selection is no longer the constraint — **candidate
recall is**.

A per-image diagnostic of the 34% of receipts with no correct candidate shows the crops are
not the problem and neither is the search:

| best digit edit-distance from any reading to the gold | share of images |
|---|---|
| 0 — read exactly | 0.656 |
| 1 | 0.110 |
| 2 | 0.102 |
| 3 | 0.093 |
| ≥5 or absent | 0.013 |

The answer is on the page and very nearly read, essentially always. The reader is simply
confidently wrong by a digit or two, which is why enlarging the search space does not help
(see §5) and why the remaining headroom is in reader accuracy itself.

`semantic_grid` is bounded differently. Its metric averages IoU over every class present in
either map, so a class predicted that the gold lacks contributes a whole zero term — and
class 2 appears in only 37% of receipts. Knowing the true class *presence* would give 0.771;
knowing the true *active set* (which cells are non-blank) would give 0.826; both, 0.864.

## 5. Reproduce

`solution.py` is fully self-contained and writes the submission CSV:

```
python3 solution.py <public_dir> <submission_out.csv>
```

On a GPU the whole run takes roughly 50–70 minutes; it auto-detects the absence of an
accelerator and shortens the schedule (lower quality, same code path). Every weight is
initialised randomly and fitted during the run — the script reads no model files, uses no
outside corpus, makes no network request, and hard-codes no per-id answers. The only external
assets are the font files already installed on the machine.

## 4. Reproduce

`solution.py` is fully self-contained and writes the submission CSV. On a GPU the whole run
takes roughly 45–60 minutes; it auto-detects the absence of an accelerator and shortens the
schedule (lower quality, same code path).

In [ ]:
PUBLIC_DIR = 'payment_summary'   # folder holding train.csv, test.csv, images/
OUT_CSV    = 'submission.csv'

import solution
solution.main(PUBLIC_DIR, OUT_CSV)

## 5. Submission check

In [ ]:
import json, pandas as pd

sub  = pd.read_csv(OUT_CSV)
test = pd.read_csv(f'{PUBLIC_DIR}/test.csv')

assert list(sub.columns) == ['id', 'answer_json']
assert len(sub) == len(test) and not sub.id.duplicated().any()
assert set(sub.id) == set(test.id)

for x in sub.answer_json:
    a = json.loads(x)
    assert set(a) == {'fields_present', 'num_items', 'semantic_grid', 'total_price'}
    assert set(a['fields_present']) == {'subtotal_present', 'service_present',
                                        'tax_present', 'etc_present'}
    assert all(isinstance(v, bool) for v in a['fields_present'].values())
    assert isinstance(a['num_items'], int) and 0 <= a['num_items'] <= 99
    g = a['semantic_grid']
    assert isinstance(g, list) and len(g) == 16
    assert all(isinstance(v, int) and 0 <= v <= 3 for v in g)
    assert isinstance(a['total_price'], str)

print('submission valid:', sub.shape)
sub.head()